# SQL Queries - Where Should You Move to Be Happy?
## Analyzing Global Happiness and GDP Across Countries

**Final Project - Group 21**

---

**This file contains ALL SQL queries used in the project, each with comments explaining:**
- What the query does
- How it works
- Why it's relevant to our research question
- What the output revealed

All queries use SQLite via Python. Data source: World Happiness Report 2024 + World Bank GDP API.

## SECTION A: DATA SETUP

In [8]:
# Import packages
import pandas as pd
import numpy as np
import sqlite3
import requests

# Load happiness data 
happiness_df = pd.read_csv('happiness_2024.csv')
happiness_df = happiness_df.drop_duplicates(subset='Country name')

# Load gdp data
gdp_df = pd.read_csv('worldbank_gdp_2024.csv')

# Create SQLite Database
conn = sqlite3.connect(':memory:')  # in-memory database for this session
cursor = conn.cursor()

# Load both datasets as SQL tables
gdp_df.to_sql('gdp_raw', conn, if_exists='replace', index=False)
happiness_df.to_sql('happiness_raw', conn, if_exists='replace', index=False)

# Create a dictionary and key to match the unmatched countries
country_map = pd.DataFrame({
    "happiness_country": [
        "South Korea",
        "Russia",
        "Turkey",
        "UAE",
        "Egypt",
        "Vietnam"
    ],
    "gdp_country": [
        "Korea, Rep.",
        "Russian Federation",
        "Turkiye",
        "United Arab Emirates",
        "Egypt, Arab Rep.",
        "Viet Nam"
    ]
})

country_map.to_sql("country_map", conn, if_exists="replace", index=False)

# Join both tables and match records using the dictionary created above
# Following analysis (refer to section B below) we noticed that some countries had different nomenclatures 
# This block of code merges the data from both tables while addressing the naming discrepancies

conn.execute("DROP VIEW IF EXISTS merged_happiness_gdp;")

query_merge = """
CREATE VIEW merged_happiness_gdp AS
SELECT 
    h.*,
    g.gdp_per_capita
FROM happiness_raw h

LEFT JOIN country_map m
    ON h."Country name" = m.happiness_country

LEFT JOIN gdp_raw g
    ON COALESCE(m.gdp_country, h."Country name") = g.country

WHERE g.gdp_per_capita IS NOT NULL;
"""

conn.execute(query_merge)

# Check shape of joined table
merged_df = pd.read_sql("""
SELECT *
FROM merged_happiness_gdp;
""", conn)

print(f'Merged dataset shape: {merged_df.shape}')

# Feature engineering - Create new columns for analysis
query_features = """
CREATE VIEW IF NOT EXISTS happiness_features AS
SELECT
    *,
    
    CASE
        WHEN gdp_per_capita >= 40000 THEN 'High Income'
        WHEN gdp_per_capita >= 15000 THEN 'Upper Middle'
        WHEN gdp_per_capita >= 5000 THEN 'Lower Middle'
        ELSE 'Low Income'
    END AS income_tier,

    LOG(gdp_per_capita) AS log_gdp,

    CASE
        WHEN "Ladder score" >= 6.0 THEN 1
        ELSE 0
    END AS is_happy,

    (
        "Social support"
        + "Freedom to make life choices"
        + ("Healthy life expectancy" / 100.0)
    ) / 3.0 AS wellbeing_score,

    RANK() OVER (
        ORDER BY gdp_per_capita DESC
    ) AS gdp_rank,

    RANK() OVER (
        ORDER BY "Ladder score" DESC
    ) AS happiness_rank

FROM merged_happiness_gdp;
"""

conn.execute(query_features)

# Check shape of engineered table
features_df = pd.read_sql("""
SELECT *
FROM happiness_features;
""", conn)

print(f'Features dataset shape: {features_df.shape}')

print('✅ Database ready')

Merged dataset shape: (62, 10)
Features dataset shape: (62, 16)
✅ Database ready


---
## SECTION B: SQL QUERIES

## Fix Unmatched Data in Merged Dataset

---
### 1. Check Unmatched Data in Happiness Dataset 
**What:** Returns the countries in the happiness dataset that were not matched when merged.

**How:** Simple SELECT, LEFT JOIN and WHERE.

**Why:** This is important to match the unmatched countries

**Output:** Six countries were identified as unmatched because of naming convention: South Korea, Russia, Turkey, UAE, Egypt, Vietnam

In [13]:
# Check unmatched countries in the happiness dataset

happiness_unmatched = pd.read_sql("""
SELECT 
    happiness_raw."Country name" AS happiness_country
FROM happiness_raw
LEFT JOIN gdp_raw
    ON happiness_raw."Country name" = gdp_raw.country
WHERE gdp_raw.country IS NULL;
""", conn)

happiness_unmatched

,happiness_country
0,South Korea
1,Russia
2,Turkey
3,UAE
4,Egypt
5,Vietnam


---
### 2. Check Unmatched Data in GDP Dataset 
**What:** Returns the countries in the GDP dataset that were not matched when merged.

**How:** Simple SELECT, LEFT JOIN,  WHERE and LIKE operations.

**Why:** This is important to match the unmatched countries

**Output:** 47 countries were identified as potential matches for the unmatched countries in the happiness database

In [14]:
# Check unmatched countries in the gdp dataset

gdp_unmatched = pd.read_sql("""
SELECT 
    gdp_raw.country AS gdp_country
FROM gdp_raw
LEFT JOIN happiness_raw
    ON gdp_raw.country = happiness_raw."Country name"
WHERE happiness_raw."Country name" IS NULL
    AND (gdp_raw.country LIKE 'S%' OR gdp_raw.country LIKE 'R%' OR gdp_raw.country LIKE 'T%' OR gdp_raw.country LIKE 'U%' OR gdp_raw.country LIKE 'U%' OR gdp_raw.country LIKE 'E%' OR gdp_raw.country LIKE 'V%' OR gdp_raw.country LIKE '%K%');
""", conn)

gdp_unmatched

,gdp_country
0,Burkina Faso
1,"Egypt, Arab Rep."
2,El Salvador
3,Equatorial Guinea
4,Estonia
5,Eswatini
6,"Hong Kong SAR, China"
7,Kiribati
8,"Korea, Rep."
9,Kosovo


## SQL Analysis

---
### Query 1 - Top 10 Happiest Countries
**What:** Returns the 10 highest-ranked countries by happiness score with their GDP and income tier.

**How:** Simple SELECT, ORDER BY Ladder score DESC, LIMIT 10.

**Why:** This is the first thing a relocating student wants to know - where are the happiest places?

**Output:** Finland (#1, 7.74), Denmark (#2, 7.58), Iceland (#3, 7.52). All Western European, North America and ANZ or Middle East and North Africa countries. All High Income.

In [15]:
pd.read_sql("""
SELECT 
    "Country name",
    "Regional indicator" AS region,
    ROUND("Ladder score", 3) AS happiness_score,
    ROUND(gdp_per_capita, 0) AS gdp_usd,
    income_tier
FROM happiness_features
ORDER BY "Ladder score" DESC
LIMIT 10;
""", conn)

,Country name,region,happiness_score,gdp_usd,income_tier
0,Finland,Western Europe,7.741,53150.0,High Income
1,Denmark,Western Europe,7.583,71026.0,High Income
2,Iceland,Western Europe,7.525,86041.0,High Income
3,Sweden,Western Europe,7.344,57117.0,High Income
4,Israel,Middle East and North Africa,7.341,54177.0,High Income
5,Netherlands,Western Europe,7.319,67520.0,High Income
6,Norway,Western Europe,7.302,86785.0,High Income
7,Luxembourg,Western Europe,7.122,137782.0,High Income
8,Switzerland,Western Europe,7.060,103998.0,High Income
9,Australia,North America and ANZ,7.057,64604.0,High Income


---
### Query 2 - Average Happiness by Region (GROUP BY)
**What:** Aggregates happiness and GDP by world region.

**How:** GROUP BY Regional indicator with AVG, MIN, MAX, COUNT.

**Why:** Students often have a region in mind — this tells them which regions offer the best happiness odds.

**Output:** Western Europe and North America/ANZ lead with avg ~6.9. South Asia lowest at ~4.3.

In [17]:
pd.read_sql("""
SELECT 
    "Regional indicator" AS region,
    COUNT(*) AS num_countries,
    ROUND(AVG("Ladder score"), 3) AS avg_happiness,
    ROUND(MIN("Ladder score"), 3) AS min_happiness,
    ROUND(MAX("Ladder score"), 3) AS max_happiness,
    ROUND(AVG(gdp_per_capita), 0) AS avg_gdp
FROM happiness_features
GROUP BY "Regional indicator"
ORDER BY avg_happiness DESC;
""", conn)

,region,num_countries,avg_happiness,min_happiness,max_happiness,avg_gdp
0,Western Europe,18,6.932,6.119,7.741,65349.0
1,North America and ANZ,4,6.926,6.725,7.057,63171.0
2,Central and Eastern Europe,4,6.306,6.000,6.822,25075.0
3,Latin America and Caribbean,10,6.006,5.345,6.609,12534.0
4,East Asia,3,5.995,5.951,6.060,27343.0
5,Southeast Asia,6,5.873,5.277,6.523,20587.0
6,Middle East and North Africa,6,5.787,4.224,7.341,27159.0
7,Commonwealth of Independent States,3,5.589,4.894,6.087,11478.0
8,Sub-Saharan Africa,5,4.714,3.966,5.635,2602.0
9,South Asia,3,4.333,4.054,4.657,2256.0


---
### Query 3 - Happiness + GDP
**What:** Show happiness alongside raw GDP.

**How:** Simple SELECT with ORDER BY and LIMIT.

**Why:** This helps us understand how the GDP of the happiest countries vary.

**Output:** Luxembourg has highest GDP ($137k) but isn't #1 in happiness.

In [19]:
pd.read_sql("""
SELECT
    "Country name",
    "Regional indicator",
    ROUND("Ladder score", 2)  AS happiness_score,
    ROUND(gdp_per_capita, 0)  AS gdp_per_capita_usd,
    income_tier
FROM happiness_features
ORDER BY "Ladder score" DESC
LIMIT 15;
""", conn)

,Country name,Regional indicator,happiness_score,gdp_per_capita_usd,income_tier
0,Finland,Western Europe,7.74,53150.0,High Income
1,Denmark,Western Europe,7.58,71026.0,High Income
2,Iceland,Western Europe,7.53,86041.0,High Income
3,Sweden,Western Europe,7.34,57117.0,High Income
4,Israel,Middle East and North Africa,7.34,54177.0,High Income
5,Netherlands,Western Europe,7.32,67520.0,High Income
6,Norway,Western Europe,7.30,86785.0,High Income
7,Luxembourg,Western Europe,7.12,137782.0,High Income
8,Switzerland,Western Europe,7.06,103998.0,High Income
9,Australia,North America and ANZ,7.06,64604.0,High Income


---
### Query 4 - JOIN: Hidden Gems (Happy but Not Rich)
**What:** Finds countries with happiness >= 6 but GDP below $30,000 - the best value destinations.

**How:** JOIN + WHERE filtering on both happiness and GDP thresholds.

**Why:** Most students can't afford Norway. This finds affordable happy places.

**Output:** Costa Rica (6.6, $18k), Brazil (6.3, $10k) Chile (6.3, $16k) are standout hidden gems.

In [21]:
pd.read_sql("""
SELECT 
    h."Country name",
    h."Regional indicator",
    ROUND(h."Ladder score", 2) AS happiness_score,
    ROUND(g.gdp_per_capita, 0) AS gdp_usd
FROM happiness_raw h
JOIN gdp_raw g ON h."Country name" = g.country
WHERE h."Ladder score" >= 6.0
  AND g.gdp_per_capita < 30000
ORDER BY h."Ladder score" DESC;
""", conn)

,Country name,Regional indicator,happiness_score,gdp_usd
0,Costa Rica,Latin America and Caribbean,6.61,18587.0
1,Brazil,Latin America and Caribbean,6.35,10311.0
2,Chile,Latin America and Caribbean,6.34,16710.0
3,Mexico,Latin America and Caribbean,6.33,14186.0
4,Uruguay,Latin America and Caribbean,6.31,23907.0
5,Romania,Central and Eastern Europe,6.26,20080.0
6,Greece,Western Europe,6.22,24626.0
7,Poland,Central and Eastern Europe,6.14,25104.0
8,Portugal,Western Europe,6.12,29292.0
9,Kazakhstan,Commonwealth of Independent States,6.09,14155.0


---
### Query 5 - GROUP BY Income Tier
**What:** Compares happiness and social metrics across income tiers.

**How:** GROUP BY the engineered income_tier column.

**Why:** Tests whether richer countries are always happier - answers the core research question.

**Output:** High income avg happiness = 7.0, Low income = 4.8. Clear relationship, but gap narrows at top.

In [23]:
pd.read_sql("""
SELECT 
    income_tier,
    COUNT(*) AS num_countries,
    ROUND(AVG("Ladder score"), 3) AS avg_happiness,
    ROUND(AVG(gdp_per_capita), 0) AS avg_gdp,
    ROUND(AVG("Social support"), 3) AS avg_social_support,
    ROUND(AVG("Freedom to make life choices"), 3) AS avg_freedom
FROM happiness_features
GROUP BY income_tier
ORDER BY avg_gdp DESC;
""", conn)

,income_tier,num_countries,avg_happiness,avg_gdp,avg_social_support,avg_freedom
0,High Income,22,7.014,69766.0,0.927,0.896
1,Upper Middle,14,6.157,26321.0,0.889,0.792
2,Lower Middle,13,5.813,10380.0,0.850,0.759
3,Low Income,13,4.844,3004.0,0.727,0.681


---
### Query 6 - WINDOW FUNCTION: Rank Within Each Region
**What:** Finds the #1 happiest country in each world region.

**How:** ROW_NUMBER() OVER (PARTITION BY region ORDER BY happiness DESC). Outer query filters rank = 1.

**Why:** Students targeting a specific region get a clear #1 recommendation.

**Output:** Finland (W. Europe), Israel (MENA), Australia (N. America/ANZ), Czechia (Central and Eastern Europe), Costa Rica (Latin America).

In [26]:
pd.read_sql("""
SELECT *
FROM (
    SELECT
        "Country name",
        "Regional indicator" AS region,
        ROUND("Ladder score", 2) AS happiness_score,
        ROUND(gdp_per_capita, 0) AS gdp_usd,
        ROW_NUMBER() OVER (
            PARTITION BY "Regional indicator"
            ORDER BY "Ladder score" DESC
        ) AS rank_in_region
    FROM happiness_features
)
WHERE rank_in_region = 1
ORDER BY happiness_score DESC;
""", conn)

,Country name,region,happiness_score,gdp_usd,rank_in_region
0,Finland,Western Europe,7.74,53150.0,1
1,Israel,Middle East and North Africa,7.34,54177.0,1
2,Australia,North America and ANZ,7.06,64604.0,1
3,Czechia,Central and Eastern Europe,6.82,31823.0,1
4,Costa Rica,Latin America and Caribbean,6.61,18587.0,1
5,Singapore,Southeast Asia,6.52,90674.0,1
6,Kazakhstan,Commonwealth of Independent States,6.09,14155.0,1
7,Japan,East Asia,6.06,32487.0,1
8,South Africa,Sub-Saharan Africa,5.63,6267.0,1
9,Pakistan,South Asia,4.66,1479.0,1


---
### Query 7 - WINDOW FUNCTION: Rolling Average Happiness by GDP Rank
**What:** Computes a rolling average of happiness as we move from poorest to richest.

**How:** AVG() OVER (ORDER BY gdp_rank ROWS BETWEEN 4 PRECEDING AND 4 FOLLOWING).

**Why:** Shows whether the GDP-happiness relationship is smooth or has a saturation point.

**Output:** Rolling average rises sharply at low GDP levels, then flattens

In [28]:
pd.read_sql("""
SELECT
    "Country name",
    ROUND(gdp_per_capita, 0) AS gdp_usd,
    gdp_rank,
    ROUND("Ladder score", 2) AS happiness_score,
    ROUND(
        AVG("Ladder score") OVER (
            ORDER BY gdp_rank
            ROWS BETWEEN 4 PRECEDING AND 4 FOLLOWING
        ), 3
    ) AS rolling_avg_happiness
FROM happiness_features
ORDER BY gdp_rank
LIMIT 20;
""", conn)

,Country name,gdp_usd,gdp_rank,happiness_score,rolling_avg_happiness
0,Luxembourg,137782.0,1,7.12,6.969
1,Ireland,112895.0,2,6.84,7.062
2,Switzerland,103998.0,3,7.06,7.014
3,Singapore,90674.0,4,6.52,7.085
4,Norway,86785.0,5,7.30,7.111
5,Iceland,86041.0,6,7.53,7.104
6,United States,84534.0,7,6.72,7.111
7,Denmark,71026.0,8,7.58,7.143
8,Netherlands,67520.0,9,7.32,7.184
9,Australia,64604.0,10,7.06,7.119


---
### Query 8 - SUBQUERY: Countries Happier Than Their Region Average
**What:** Finds countries that outperform their regional average - overachievers worth targeting.

**How:** JOIN to a subquery that calculates regional averages. Filter where country > region average.

**Why:** Identifies countries punching above their weight - often the best relocation targets.

**Output:** Israel and UAE (+1.55 and +0.95 above MENA avg respectively) are top overperformers.

In [30]:
pd.read_sql("""
SELECT 
    h."Country name",
    h."Regional indicator" AS region,
    ROUND(h."Ladder score", 2) AS happiness_score,
    ROUND(region_avg.avg_score, 2) AS region_avg_happiness,
    ROUND(h."Ladder score" - region_avg.avg_score, 2) AS above_region_avg
FROM happiness_features h
JOIN (
    -- Subquery: calculate average happiness per region
    SELECT 
        "Regional indicator",
        AVG("Ladder score") AS avg_score
    FROM happiness_features
    GROUP BY "Regional indicator"
) AS region_avg
ON h."Regional indicator" = region_avg."Regional indicator"
WHERE h."Ladder score" > region_avg.avg_score
ORDER BY above_region_avg DESC
LIMIT 15;
""", conn)

,Country name,region,happiness_score,region_avg_happiness,above_region_avg
0,Israel,Middle East and North Africa,7.34,5.79,1.55
1,UAE,Middle East and North Africa,6.73,5.79,0.95
2,South Africa,Sub-Saharan Africa,5.63,4.71,0.92
3,Finland,Western Europe,7.74,6.93,0.81
4,Singapore,Southeast Asia,6.52,5.87,0.65
5,Denmark,Western Europe,7.58,6.93,0.65
6,Costa Rica,Latin America and Caribbean,6.61,6.01,0.60
7,Iceland,Western Europe,7.53,6.93,0.59
8,Saudi Arabia,Middle East and North Africa,6.32,5.79,0.54
9,Czechia,Central and Eastern Europe,6.82,6.31,0.52


---
### Query 9 - CTE SUBQUERY: Bottom 10 Countries to Avoid
**What:** Identifies the 10 least happy countries - places students should NOT relocate to.

**How:** WITH clause (CTE) ranks all countries by happiness ascending. Outer query filters bottom 10.

**Why:** Just as important as knowing where to go is knowing where NOT to go.

**Output:** Ethiopia (3.97) India (4.05), Egypt (4.22) rank lowest in our dataset.

In [32]:
pd.read_sql("""
WITH ranked_countries AS (
    -- CTE: rank all countries by happiness score
    SELECT
        "Country name",
        "Regional indicator",
        ROUND("Ladder score", 2) AS happiness_score,
        ROUND(gdp_per_capita, 0) AS gdp_usd,
        ROW_NUMBER() OVER (ORDER BY "Ladder score" ASC) AS bottom_rank
    FROM happiness_features
)
SELECT *
FROM ranked_countries
WHERE bottom_rank <= 10
ORDER BY happiness_score ASC;
""", conn)

,Country name,Regional indicator,happiness_score,gdp_usd,bottom_rank
0,Ethiopia,Sub-Saharan Africa,3.97,1134.0,1
1,India,South Asia,4.05,2695.0,2
2,Egypt,Middle East and North Africa,4.22,3338.0,3
3,Bangladesh,South Asia,4.29,2593.0,4
4,Nigeria,Sub-Saharan Africa,4.48,1084.0,5
5,Kenya,Sub-Saharan Africa,4.58,2132.0,6
6,Turkey,Middle East and North Africa,4.61,15893.0,7
7,Pakistan,South Asia,4.66,1479.0,8
8,Ukraine,Commonwealth of Independent States,4.89,5389.0,9
9,Ghana,Sub-Saharan Africa,4.91,2391.0,10


---
### Query 10 - JOIN + GROUP BY + HAVING: Freedom vs Happiness by Region
**What:** Compares personal freedom and happiness across regions, filtering to regions with 3+ countries.

**How:** JOIN happiness to GDP + GROUP BY region + HAVING COUNT >= 3.

**Why:** Tests whether freedom (not just money) drives happiness. Relevant to students valuing personal liberty.

**Output:** Western Europe has both happiness (6.9) and highest freedom (0.87) as well as North America and ANZ with happiness (6.9) and highest freedom (0.90) scores. Freedom and happiness highly correlated.

In [34]:
pd.read_sql("""
SELECT
    h."Regional indicator" AS region,
    COUNT(*) AS countries,
    ROUND(AVG(h."Ladder score"), 3) AS avg_happiness,
    ROUND(AVG(h."Freedom to make life choices"), 3) AS avg_freedom,
    ROUND(AVG(g.gdp_per_capita), 0) AS avg_gdp
FROM happiness_raw h
JOIN gdp_raw g ON h."Country name" = g.country
GROUP BY h."Regional indicator"
HAVING COUNT(*) >= 3
ORDER BY avg_happiness DESC;
""", conn)

,region,countries,avg_happiness,avg_freedom,avg_gdp
0,Western Europe,18,6.932,0.877,65349.0
1,North America and ANZ,4,6.926,0.902,63171.0
2,Middle East and North Africa,3,6.384,0.781,31151.0
3,Central and Eastern Europe,4,6.306,0.812,25075.0
4,Latin America and Caribbean,10,6.006,0.795,12534.0
5,Southeast Asia,5,5.933,0.804,23761.0
6,Sub-Saharan Africa,5,4.714,0.682,2602.0
7,South Asia,3,4.333,0.552,2256.0


---
### Query 11 - ROLLUP-Style: Happiness by Income Tier with Grand Total
**What:** Shows happiness stats per income tier PLUS a grand total row - simulates SQL ROLLUP.

**How:** UNION ALL combines per-tier GROUP BY with a global aggregate.

**Why:** Gives a complete picture of happiness distribution with a baseline for comparison.

**Output:** Grand total avg happiness = 6.11. High income avg = 7.0 - confirms wealthy countries are happier on average.

In [36]:
pd.read_sql("""
SELECT 
    income_tier AS group_label,
    'Income Tier' AS group_type,
    COUNT(*) AS countries,
    ROUND(AVG("Ladder score"), 3) AS avg_happiness,
    ROUND(AVG(gdp_per_capita), 0) AS avg_gdp
FROM happiness_features
GROUP BY income_tier

UNION ALL

-- Grand total (simulates ROLLUP grand total row)
SELECT 
    'ALL COUNTRIES' AS group_label,
    'Grand Total' AS group_type,
    COUNT(*) AS countries,
    ROUND(AVG("Ladder score"), 3) AS avg_happiness,
    ROUND(AVG(gdp_per_capita), 0) AS avg_gdp
FROM happiness_features
ORDER BY avg_happiness DESC;
""", conn)

,group_label,group_type,countries,avg_happiness,avg_gdp
0,High Income,Income Tier,22,7.014,69766.0
1,Upper Middle,Income Tier,14,6.157,26321.0
2,ALL COUNTRIES,Grand Total,62,6.113,33505.0
3,Lower Middle,Income Tier,13,5.813,10380.0
4,Low Income,Income Tier,13,4.844,3004.0
